In [14]:
%matplotlib inline
import matplotlib.pyplot as plt
import jax.numpy as jnp
import numpy as np
from jax import random
import seaborn as snb

from bayesian import *

# use 64-bit precision
from jax import config
config.update("jax_enable_x64", True)

# style stuff
snb.set_style('darkgrid')
snb.set_theme(font_scale=1.25)
colors = snb.color_palette()

# 02477 Bayesian Machine Learning Exam Template

# Part 1
Solution to Q1.1 through Q1.3: See handwritten solution on appendix A

In [15]:
# Part 1

# We apply the results shown in appendix A and print the mean vectors and covariance matrices of the distributions.
A = jnp.array([0, 1, 1, 0]).reshape((2,2))
b = jnp.ones((2,))
W = jnp.array([1, 1, 1, 0]).reshape((2,2))

# Q1.2
Sigma_xz = jnp.linalg.inv(jnp.eye(2) + 2*A.T @ A)
# print(f'p(z|x): mu = {:3.5f}, Sigma = {jnp.eye(2) + 2*A.T @ A}')
# Maybe look into printing it with sympy?

# Part 2
Solution to Q2.1 through Q2.5: See handwritten solution on appendix B

In [61]:
# Part 2
# Define problem variables
X = jnp.array([0,0,1,1]).T.reshape((2,2))
y = jnp.array([0,1])
w_map = jnp.array([1,1])*0.74077439
x_star = jnp.arange(2)
lambda1 = 1/4

#plot_data(X, y)

# Q2.1 Apply the plugin approximation using the given map estimate.
from bayesian import sigmoid
p = sigmoid(w_map @ x_star)
print(f'p(y*|y, x*): Ber({p:3.5f})') 
# This checks out when looking at the dataset

# Q2.2
log_npdf = lambda x, m, v: -(x-m)**2/(2*v) - 0.5*jnp.log(2*jnp.pi*v) # v is the variance
npdf = lambda x, m, v: jnp.exp(log_npdf(x, m, v))
# Evaluate the log joint for the given dataset, i.e. over all datapoints.


p = sigmoid(X @ w_map)
bernoulli_logpmf(y, p)
log_p_yw = bernoulli_logpmf(y, p).sum() + log_npdf(w_map, 0, lambda1**-1).sum()
print(f'log p(y,w) =  {log_p_yw:3.5f})') 

#Q2.3

from jax import hessian
from scipy.optimize import minimize


neg_log_lik = lambda w: -(bernoulli_logpmf(y, sigmoid(X @ w)).sum() + log_npdf(w, 0, lambda1**-1).sum())
w_map_true = minimize(neg_log_lik, jnp.zeros(2))
#print(w_map_true)
log_lik = lambda w: (bernoulli_logpmf(y, sigmoid(X @ w)).sum() + log_npdf(w, 0, lambda1**-1).sum())


pos_hessian = hessian(log_lik)(w_map)

p(y*|y, x*): Ber(0.67717)
log p(y,w) =  -4.25931)


In [ ]:
# Q2.4
# First we determine the laplace approximation.
m, S = laplace_approximation(log_lik, jnp.zeros(2))
m = w_map 
S = -jnp.linalg.inv(pos_hessian) 
# Since the posterior predictive distribution is intractable for this model, we can either sample from the approximate posterior nd make a monte carlo estimate or
# we can apply the probit approximation. For now we apply the probit approximation.

# compute p(f^*|y, x^*)
fstar_mean = x_star @ w_map
fstar_var = x_star.T@S@x_star
print(f'p(f^*|y) = N(f^*|{fstar_mean:3.2f}, {fstar_var:3.2f})')
# compute p(y^*|y, x^*)
import scipy
Phi = scipy.stats.norm.cdf
p_ystar1 = Phi(fstar_mean/np.sqrt(8/np.pi + fstar_var))
print(f'p(y^* = 1|y, x^*=[0, 1].T) = {p_ystar1:3.2f} (probit approximation)')

# Q2.5

p(f^*|y) = N(f^*|0.74, 2.91)
p(y^* = 1|y, x^*=-3) = 0.62 (probit approximation)


# Q2.5
First we note that the new target variable $z \in \mathbb{N}$ and as such this is a natural number regression task. 
In this case I would change the likelyhood to follow a poisson distribution parameterized by x(n)@w. I would  keep the same prior on the weights.
One could consider making the model more complex by mapping x through some feature transformation to include second and higher order dependencies
in the regression model.  

# Part 3
Solution to Q3.1 through Q3.5: See handwritten solution on appendix C

In [ ]:
# Part 3


# Part 4
Solution to Q4.1 through Q4.5: See handwritten solution on appendix D

In [ ]:
# Part 4
m = jnp.array([-0.87, 2.13])
S = jnp.array([0.41, 0.11, 0.11, 0.27]).reshape((2,2))

# Q4.1
# By the symmetry of prior distribution of w1 around the mean w1=0, easily see that
# the prior probability p(w1>0) = 0.5
# To compute the posterior probability, we compute the quantiles for the posterior destriubution of w1
scipy.stats.norm.cdf(0, loc = m[0], scale = S[0,0]) 
# This is makes sense as the 0 is about 2 standard deviations out form the mean.
print(f'p(w < 0|y) = {(1- scipy.stats.norm.cdf(0, loc = m[0], scale = S[0,0]) ):3.4f} ')

# Q4.2
print(f' 80% credibility interval for p(w2|y): {scipy.stats.norm.interval(confidence=0.80, loc = m[1], scale = S[1,1])}')

# Q4.3
x_star = 1
# This is a linear transformation of gaussians, thus f* is also a gaussian. We apply the equations from Murphy 1 3.3
A = jnp.ones(2)
mu = A @ m
Sigma = A @ S @ A.T
print(f'p(f*|y, x*): mu = {mu:3.5f}, Sigma = {Sigma}')

# Q4.4
# This is the same just with added measurement noise. Agian we refer to Murphy 3.3 for linear systems of Gaussians
sigma2 = jnp.sqrt(2)
print(f'p(f*|y, x*): mu = {mu:3.5f}, Sigma = {Sigma + sigma2}')






p(w < 0|y) = 0.0169 
 80% credibility interval for p(w2|y): (np.float64(1.7839810773029579), np.float64(2.476018922697042))
p(f*|y, x*): mu = 1.26000, Sigma = 0.9


# Part 5
Solution to Q5.1 through Q5.5: See handwritten solution on appendix E

In [19]:
# Part 5